# 第 5 章 — ML ポテンシャル (MACE-MP-0) で同じ TS を解く

**ゴール**
- ASE calculator のすげ替えで、xTB → ML ポテンシャルに 1 行で乗り換える
- 同じ HCN ⇌ HNC 反応を **MACE-MP-0** で解いて、**2 章の xTB の結果と比較** する
- 4 章で学んだ拘束付き TS 探索を MACE で再実行する (章間の伏線回収)

## 事前準備

```bash
pip install mace-torch
```

**初回ダウンロードの注意**: `mace_mp(...)` の初回呼び出し時にチェックポイント (数百 MB) が `~/.cache/mace/` にダウンロードされます。プロキシ環境では `HTTPS_PROXY` を設定するか、事前に手動で取得してください。

GPU があれば `device='cuda'`、CPU の場合は `device='cpu'` を指定します。小さな分子なら CPU でも 1 ステップ 1 秒以下で実用的です。

In [ ]:
from mace.calculators import mace_mp

# 'small' が最軽量。'medium' / 'large' でより精度が上がる代わりに遅くなる
def mace(model='small'):
    return mace_mp(
        model=model,
        default_dtype='float64',
        device='cpu',
        dispersion=False,
    )

## 2 章の xTB エネルギーを取り戻す

2 章の末尾で `%store` した値を読み戻します。**もし下のセルで `NameError` が出たら、先に 2 章のノートブックを最後まで実行してください**。

In [ ]:
%store -r E_hcn_xtb E_hnc_xtb E_ts_xtb
print(f'xTB: HCN={E_hcn_xtb:.5f}, HNC={E_hnc_xtb:.5f}, TS={E_ts_xtb:.5f} eV')

## MACE で HCN / HNC を最小化

In [ ]:
import numpy as np
from ase import Atoms
from sella import Sella

def hcn_geometry(angle_deg, r_cn=1.17, r_ch=1.10):
    theta = np.deg2rad(180.0 - angle_deg)
    H = [-r_ch * np.cos(theta), r_ch * np.sin(theta), 0.0]
    return Atoms('HCN', positions=[H, [0,0,0], [r_cn, 0, 0]])

hcn = hcn_geometry(180.0); hcn.calc = mace()
Sella(hcn, order=0, logfile=None).run(fmax=1e-3, steps=300)
E_hcn_mace = hcn.get_potential_energy()

hnc = hcn_geometry(0.0); hnc.calc = mace()
Sella(hnc, order=0, logfile=None).run(fmax=1e-3, steps=300)
E_hnc_mace = hnc.get_potential_energy()

print(f'HCN (MACE-MP-0) : {E_hcn_mace:.5f} eV')
print(f'HNC (MACE-MP-0) : {E_hnc_mace:.5f} eV')
print(f'ΔE (HNC - HCN)  : {(E_hnc_mace - E_hcn_mace)*1000:.1f} meV')

## MACE で TS 探索

2 章とまったく同じ手順 (`order=1`)、calculator だけ差し替え。

In [ ]:
ts_guess = hcn_geometry(angle_deg=90.0)
ts_guess.calc = mace()

opt = Sella(ts_guess, order=1, trajectory='ts_mace.traj', logfile='ts_mace.log')
opt.run(fmax=1e-3, steps=500)

E_ts_mace = ts_guess.get_potential_energy()
print(f'TS (MACE-MP-0) : {E_ts_mace:.5f} eV')
print(f'Ea (HCN→TS, MACE) = {(E_ts_mace - E_hcn_mace):.3f} eV')
print(f'Ea (HNC→TS, MACE) = {(E_ts_mace - E_hnc_mace):.3f} eV')

## xTB と MACE の比較プロット

**絶対エネルギー** は手法によって基準が異なるため比較できませんが、HCN を基準にした **相対値 (障壁高さ・反応エンタルピー)** は揃って比較できます。

In [ ]:
import matplotlib.pyplot as plt

labels = ['HCN', 'TS', 'HNC']
E_xtb_rel  = [0.0, E_ts_xtb  - E_hcn_xtb,  E_hnc_xtb  - E_hcn_xtb]
E_mace_rel = [0.0, E_ts_mace - E_hcn_mace, E_hnc_mace - E_hcn_mace]

x = [0, 1, 2]
plt.figure(figsize=(6, 4))
plt.plot(x, [e*1000 for e in E_xtb_rel],  'o-', label='GFN2-xTB')
plt.plot(x, [e*1000 for e in E_mace_rel], 's-', label='MACE-MP-0')
plt.xticks(x, labels)
plt.ylabel('ΔE from HCN [meV]'); plt.title('HCN ⇌ HNC: xTB vs MACE-MP-0')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f'Ea(HCN→TS): xTB = {(E_ts_xtb - E_hcn_xtb)*1000:.0f} meV, MACE = {(E_ts_mace - E_hcn_mace)*1000:.0f} meV')
print(f'ΔE(HNC-HCN): xTB = {(E_hnc_xtb - E_hcn_xtb)*1000:.0f} meV, MACE = {(E_hnc_mace - E_hcn_mace)*1000:.0f} meV')

## ベンチマーク (簡易): 力評価 1 回あたりの時間

In [ ]:
import time
from tblite.ase import TBLite

for label, calc in [('xTB',  TBLite(method='GFN2-xTB', verbosity=0)),
                    ('MACE', mace())]:
    geo = hcn_geometry(90.0); geo.calc = calc
    _ = geo.get_forces()                # ウォームアップ
    t0 = time.time()
    for _ in range(20):
        geo.calc.results.clear()
        _ = geo.get_forces()
    dt = (time.time() - t0) / 20
    print(f'{label:5}: {dt*1000:7.1f} ms / force call')

## 演習

1. `mace_mp(model='medium')` に変えると TS エネルギーがどう変わるか比較してください (CPU では数倍遅くなります)。
2. IRC (3 章) を MACE で流して、xTB の経路と重ねてプロットしてみましょう。経路の形が手法でどれくらい変わるかが見えます。
3. **拘束付き TS (4 章) を MACE で取り直す**: 4 章の最後で得た「C-N を 1.17 Å に固定した拘束付き TS」を MACE でも再現してください。calculator だけ差し替えれば動くはずです。
4. MACE で **Cu4 (1 章の系)** を最小化できるか試してください。MACE-MP-0 は周期表全域をカバーする「Foundation Model」なので、EMT の代替にもなり得ます。**ヒント**: small で収束しない場合は `model='medium'` に上げてみてください (金属系では精度差が大きいことがあります)。

---
## まとめ

- Sella のコード本体は calculator に **一切依存しません**。`atoms.calc` を差し替えるだけで EMT / xTB / MACE を行き来できます。
- TS 探索の品質は **(1) 初期推定の良さ**, **(2) PES の滑らかさ**, **(3) ハイパーパラメータ** の 3 つで決まります。
- ML ポテンシャルは「とりあえずまともな PES が欲しい」場面で xTB と並ぶ強い選択肢。Sella と組み合わせると探索もそのまま回せます。

**次にチャレンジできること**:
- 自分の関心反応 (Diels-Alder、求核置換 SN2 など) の TS を、まず xTB or MACE で当たりをつけ、必要なら DFT で精緻化する
- ASE の `NEB` で大まかな経路を作ってから、各イメージを `order=1` で Sella に渡して TS を絞り込む
- 触媒表面 (Sella 公式の README サンプル) — 気相で慣れたら自然に表面拡張できます